# NASA EPSCoR Progress: Sénarmont QWP Alignment for EOM Bias
### 20261104 - *Tyndale Stutzman*
---

## Introduction:

One-off procedure. The [Thorlabs EO-PM-NR-C1](https://www.thorlabs.com/item/EO-PM-NR-C1) fringe did not fit a peak *and* a trough inside our drive window, so $V_\pi$ could not be read with a ruler (run `sine_20261104T181902Z`: peak at −0.20 V tee, still falling at +0.49 V). A quarter-wave plate after the EOM (a Sénarmont compensator) lets the analyzer HWP slide the fringe along the drive axis. This notebook aligns that QWP with live Red Pitaya views and then hands back to the EOM notebook for the $V_\pi$ run.

Only one program can own the Red Pitaya: shut down the EOM notebook's kernel before connecting here, and vice versa.

## Table of Contents

1. [Background](#background)
1. [Setup](#setup)
   1. [Environment & Packages](#env)
   1. [Parameter Configuration](#conf)
   1. [Live View](#live)
1. [Alignment](#align)
   1. [Dark Level](#dark)
   1. [Null Hunt](#null)
   1. [Slide Test](#slide)
1. [Record (2026-11-04)](#record)

<a id="background"></a>
## Background

**The fringe.** For a retarder between polarizers (input at $\alpha$, analyzer at $\beta$ to the crystal axes),
$$I(V) = A + B\cos\!\left(\Gamma_0 + \frac{\pi V}{V_\pi}\right),\qquad A = \cos^2\!\alpha\cos^2\!\beta + \sin^2\!\alpha\sin^2\!\beta,\quad B = \tfrac12\sin 2\alpha\,\sin 2\beta .$$
Peaks sit at $V_k = (2k\pi - \Gamma_0)\,V_\pi/\pi$ and troughs halfway between: where they fall on the drive axis is fixed by the static retardance $\Gamma_0$, not by us.

**Why a DC offset does not help.** The drive window is fixed: $|V_\text{dc}| + V_\text{peak} \le V_\text{max}$, set by the 3211 (±200 V, gain 40, linear input ±5 V) and our command limit (0.48 V ≈ ±0.49 V tee). An offset only slides a *smaller* sweep around inside that same window; it cannot reach a voltage the full sweep could not. With $V_\pi \approx 0.80$ V tee (fitted, 2026-11-04) and a window 0.98 V wide, a peak–trough pair fits only for some values of $\Gamma_0$. On 2026-11-04 the trough sat at ≈ +0.60 V tee, outside. So we must change $\Gamma_0$ itself, optically.

**Why the analyzer alone does not help.** In the formula above the angles enter only $A$ and $B$. Rotating the analyzer changes contrast, or flips the sign of $B$ (peak ↔ trough at the same $V$), but never slides the fringe.

**Sénarmont.** Input linear at 45° to the crystal axes $(x, y)$, EOM retardance $\Gamma$:
$$\mathbf E_\text{in} = \tfrac{1}{\sqrt2}\begin{pmatrix}1\\1\end{pmatrix} \;\xrightarrow{\text{EOM}}\; \tfrac{1}{\sqrt2}\begin{pmatrix}e^{i\Gamma/2}\\ e^{-i\Gamma/2}\end{pmatrix}.$$
In the ±45° basis $\hat u = (\hat x+\hat y)/\sqrt2$, $\hat w = (\hat x-\hat y)/\sqrt2$:
$$E_u = \tfrac12\left(e^{i\Gamma/2}+e^{-i\Gamma/2}\right) = \cos\tfrac{\Gamma}{2},\qquad E_w = \tfrac12\left(e^{i\Gamma/2}-e^{-i\Gamma/2}\right) = i\sin\tfrac{\Gamma}{2}.$$
The two components are 90° out of phase: an ellipse with axes along $\hat u, \hat w$. A QWP with its fast axis along $\hat u$ removes the $i$:
$$\mathbf E = \cos\tfrac{\Gamma}{2}\,\hat u + \sin\tfrac{\Gamma}{2}\,\hat w,$$
**linear** light at angle $\Gamma/2$ from $\hat u$. Through an analyzer at $\theta$ from $\hat u$,
$$I = I_0\cos^2\!\left(\tfrac{\Gamma}{2} - \theta\right) = \tfrac{I_0}{2}\left[1 + \cos(\Gamma - 2\theta)\right].$$
Rotating the analyzer by $\theta$ shifts the bias by $2\theta$ at full contrast. Our analyzer is a HWP in front of a fixed PBS; turning a HWP by $\varphi$ turns the polarization by $2\varphi$, so the bias shifts by $4\varphi$, and the fringe slides along the drive axis by $|\Delta V| = 4\varphi\,V_\pi/\pi$ ($\varphi$ in radians).

**What to see on the bench.**
1. *Null:* only with the QWP along $\hat u$ (or $\hat w$) is the light linear, so only then can the analyzer bring the PD down to dark. A misaligned QWP leaves an ellipse: the minimum stays above dark.
2. *Slide without shrink:* with a sine drive, rotating the analyzer slides the XY fringe sideways; peak and trough levels stay put. If the fringe gets shallower as it slides, the QWP (or its retardance) is off.
3. *Frequency doubling:* biased at a peak or trough, a small sine gives PD at $2f$; at a halfway crossing, at $f$.

**Assumptions** (and the check): QWP is quarter-wave at 830 nm (slide without shrink); input polarization at 45° to the crystal (near-dark null); fast-axis choice $\hat u$ vs $\hat w$ only flips the slide direction; $\Gamma_0$ keeps drifting (re-centre before each $V_\pi$ run).

<a id="setup"></a>
## Setup

<a id="env"></a>
### Environment & Packages

Run from `notebooks/` in the repository environment.

In [ ]:
import sys
from pathlib import Path
from importlib.metadata import version

src = (Path("python") if Path("python").is_dir() else Path("../python")).resolve()
assert (src / "instruments").is_dir(), "Start in the repo root or notebooks/"
for directory in (src, src.parent):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

print(f"Python {sys.version.split()[0]} | {sys.executable}")
print({name: version(name) for name in ("numpy", "pyrpl")})

<a id="conf"></a>
### Parameter Configuration

Typed by hand: connection, jumpers, drive limits and the bench log. Same limits as the EOM notebook's sine stage.

**Wiring:** OUT1 → BNC tee → preamp → 3211 → EOM, and tee → IN2 (LV). PD → IN1 (LV).
**Optics:** input HWP → EOM → QWP → analyzer HWP → PBS → PD.

In [ ]:
rp_hostname = "192.168.1.98"
rp_config = "scope_config"
ssh_host_key_sha256 = "W5g8Qcvu7r+8mMQZInHzNLpvOpYsvnCvo5hwYkWJCRY"
results_directory = src.parent / "experiments" / "_local"

rp_jumpers = {"in1": "LV", "in2": "LV"}
rp_full_scale_v = {k: {"LV": 1.0, "HV": 20.0}[v] for k, v in rp_jumpers.items()}

# Drive limits: as the EOM notebook (0.48 V command ≈ 167-195 V at the EOM, gain unverified with this EOM).
command_limit_v = 0.48
frequency_limits_hz = (0.5, 2.0)
targets_tee_v = (-0.40, +0.40)  # where we want the peak and the trough

# Bench log: typed by hand, with source. Fill every None before connecting.
bench = {
    "eom": "Thorlabs EO-PM-NR-C1",
    "wavelength_nm": 830.0,
    "input_hwp_deg": None,      # dial
    "qwp_deg": None,            # dial
    "analyzer_hwp_deg": None,   # dial (changes during the slide test: type the start value)
    "tec": None,                # setpoint / reading
    "preamp_and_3211": "preamp x8.89 (operator) -> New Focus 3211, bias knob zeroed",
    "pd_gain": "PDA100A2 at 0 dB",
    "analog_check": None,       # what the scope shows
}
print(f"unfilled bench log: {[k for k, v in bench.items() if v is None] or 'none'}")

<a id="live"></a>
### Live View

Plumbing. `live(mode, seconds=..., note=...)` owns the Red Pitaya for one run and redraws after every buffer:
- `"hold"`: OUT1 held at 0 V; strip chart of the PD mean per 134 ms buffer, with the dark level if given.
- `"sine"`: OUT1 sine at 0 V offset (default 0.48 V peak, 2 Hz); raw XY of PD vs tee per 2.1 s buffer, target lines, and markers on the peak and trough.

Stop early with the Jupyter ■ button: the drive session returns OUT1 to zero/off either way. Raw counts are saved (every sine buffer; every 10th hold buffer, or all with `keep_every=1`) with a per-buffer log in `metadata.json`.

The peak/trough markers come from 20 mV bins of tee (Decision A1). That is a steering aid only; no number from this notebook is a $V_\pi$.

In [ ]:
import json, subprocess, time
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from instruments.red_pitaya import RedPitaya

def bin_extrema(pd, tee, width=0.02):
    '''Decision A1 (display aid): mean PD in 20 mV tee bins; is the max/min away from the two end bins?'''
    idx = np.digitize(tee, np.arange(tee.min(), tee.max() + width, width))
    bins = [i for i in np.unique(idx) if (idx == i).sum() >= 5]
    c = np.array([tee[idx == i].mean() for i in bins]); mu = np.array([pd[idx == i].mean() for i in bins])
    inside = lambda i: 2 <= i <= len(c) - 3
    return c, mu, mu.argmax(), mu.argmin(), inside(mu.argmax()), inside(mu.argmin())

def draw(mode, pd, tee, log, dark_v):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    if mode == "hold":
        t, m = np.array([[r["t_s"], r["pd_mean_v"]] for r in log]).T * [[1], [1e3]]
        ax.plot(t, m, ".-")
        ax.set(xlabel="Time (s)", ylabel="PD mean per buffer, raw (mV)",
               title=f"HOLD OUT1 = 0 V | PD now {m[-1]:.1f} mV | lowest {m.min():.1f} mV")
    else:
        c, mu, ip, it, pin, tin = bin_extrema(pd, tee)
        r = log[-1]
        ax.plot(tee, pd, ",", color="k", label="raw, this buffer")
        ax.plot(c, mu, color="tab:orange", lw=1, label="20 mV bins (A1, steering aid)")
        ax.plot(c[ip], mu[ip], "v", color="tab:red", ms=10); ax.plot(c[it], mu[it], "^", color="tab:blue", ms=10)
        for x in targets_tee_v:
            ax.axvline(x, color="tab:green", ls="--", lw=1)
        ax.set(xlim=(-0.52, 0.52), xlabel="OUT1 at tee, IN2 raw (V)", ylabel="PD, IN1 raw (V)")
        ax.set_title(f"{r['verdict']}\npeak {r['peak_tee_v']:+.3f} V ({r['peak_pd_v']*1e3:.1f} mV)   "
                     f"trough {r['trough_tee_v']:+.3f} V ({r['trough_pd_v']*1e3:.1f} mV)",
                     color="tab:green" if r["verdict"].startswith("HIT") else "k")
        ax.legend(loc="lower left", fontsize=8, markerscale=20)
    if dark_v is not None:
        ax.axhline(dark_v * (1e3 if mode == "hold" else 1), color="k", ls=":", lw=1, label="dark")
    clear_output(wait=True); display(fig); plt.close(fig)

def live(mode, *, seconds, note, peak_v=command_limit_v, frequency_hz=2.0, dark_v=None, keep_every=None):
    '''Own the RP for `seconds`, redraw every buffer, save raw counts and a per-buffer log. Returns the run folder.'''
    missing = [k for k, v in bench.items() if v is None]
    assert not missing and note, f"Fill the bench log and type a note first: {missing}"
    keep_every = keep_every or (1 if mode == "sine" else 10)
    run = results_directory / f"qwp_align_{mode}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"
    run.mkdir(parents=True)
    git = lambda *a: subprocess.run(["git", "-C", str(src.parent), *a], capture_output=True, text=True).stdout.strip()
    record = {"status": "running", "mode": mode, "note": note, "bench": dict(bench), "full_scale_v": rp_full_scale_v,
              "drive": {"peak_v": peak_v if mode == "sine" else 0.0, "frequency_hz": frequency_hz if mode == "sine" else None,
                        "command_limit_v": command_limit_v}, "dark_v": dark_v,
              "started_utc": datetime.now(timezone.utc).isoformat(),
              "git": {"commit": git("rev-parse", "HEAD"), "dirty": git("status", "--porcelain")},
              "calling_cell": globals().get("_ih", ["Not available outside IPython"])[-1],
              "captures": [], "log": []}
    save = lambda: (run / "metadata.json").write_text(json.dumps(record, indent=1, default=str))
    raw = {}
    drive = RedPitaya.drive_session(rp_hostname, profile=rp_config, ssh_host_key_sha256=ssh_host_key_sha256,
        command_limits_v={"out1": [-command_limit_v, command_limit_v]}, frequency_limits_hz=list(frequency_limits_hz),
        max_peak_v=command_limit_v, frequency_correction=1.0)
    try:
        with drive:
            if mode == "hold":
                record["command"] = drive.set_waveform("out1", waveform="dc", offset_v=0.0, peak_v=0, frequency_hz=0)
                decimation = 1024      # 134 ms buffers
            else:
                record["command"] = drive.set_waveform("out1", offset_v=0.0, peak_v=peak_v, frequency_hz=frequency_hz, ramp_s=0.5)
                decimation = 16384     # 2.15 s buffers
            time.sleep(1.0)
            t0 = time.monotonic()
            while time.monotonic() - t0 < seconds:
                cap = drive.capture(decimation=decimation, full_scale_v=rp_full_scale_v)
                pd, tee = cap["input_volts"]
                n = len(record["log"])
                row = {"name": f"c{n:04d}", "t_s": time.monotonic() - t0, "pd_mean_v": pd.mean(), "pd_std_v": pd.std(),
                       "issues": cap["metadata"]["issues"]}
                if mode == "sine":
                    c, mu, ip, it, pin, tin = bin_extrema(pd, tee)
                    row.update(peak_tee_v=c[ip], peak_pd_v=mu[ip], trough_tee_v=c[it], trough_pd_v=mu[it],
                               verdict="HIT: peak and trough both inside" if pin and tin else
                               f"peak {'inside' if pin else 'AT EDGE'}, trough {'inside' if tin else 'AT EDGE'}")
                record["log"].append(row)
                if n % keep_every == 0:
                    raw[row["name"]] = cap["raw_counts"]
                    record["captures"].append({"name": row["name"], "metadata": cap["metadata"]})
                if n % 10 == 0:
                    np.savez_compressed(run / "raw.npz", **raw); save()
                draw(mode, pd, tee, record["log"], dark_v)
        record["status"] = "completed"
    except KeyboardInterrupt:
        record["status"] = "stopped by operator"
    except BaseException as error:
        record.update(status="failed", error=f"{type(error).__name__}: {error}")
        raise
    finally:
        record.update(cleanup=drive.cleanup_evidence, finished_utc=datetime.now(timezone.utc).isoformat())
        if raw:
            np.savez_compressed(run / "raw.npz", **raw)
        save()
        print(f"{record['status']}; {len(record['log'])} buffers; cleanup={drive.cleanup_evidence.get('status')}; saved to {run}")
    return run

<a id="align"></a>
## Alignment

<a id="dark"></a>
### Dark Level

**Question:** What does IN1 read with no light? This is the floor a perfect null would reach.

**Prediction:** A few mV negative (2026-11-04: −6.0 to −6.4 mV at PD 0 dB).

**How:** Block the beam right at the PD for the whole run.

**Notable Decisions:**
- (A2) Dark = mean of the saved raw buffers of a fully blocked run.

In [ ]:
dark_run = live("hold", seconds=20, keep_every=1, note="beam blocked at the PD for the whole run")

In [ ]:
# Dark level (Decision A2): mean of every saved raw buffer.
with np.load(dark_run / "raw.npz") as z:
    dark_v = float(np.mean([z[k][0].mean() for k in z.files])) / 8192 * rp_full_scale_v["in1"]
    spread = np.ptp([z[k][0].mean() for k in z.files]) / 8192 * rp_full_scale_v["in1"]
print(f"dark = {dark_v*1e3:.2f} mV (buffer means spread {spread*1e3:.2f} mV) from {dark_run.name}")

<a id="null"></a>
### Null Hunt

**Question:** Is there a QWP angle that makes the light after it linear (so the analyzer can null it)?

**Prediction:** Yes; at that angle the minimum over analyzer angle reaches within a few mV of dark.

**How (no drive, OUT1 = 0 V):**
1. Turn the analyzer HWP slowly through ≤ 45° and find the PD minimum.
2. Nudge the QWP ~2°; keep the direction that makes the minimum deeper.
3. Repeat 1–2 until nudging either knob only raises the PD.

**What could fool us:**
- Lit-level drift (≈ 3 mV in 20 s on 2026-11-04): judge by the depth of the dip, not by small wobbles.
- If no clear minimum appears while turning the analyzer, the light is near circular: turn the QWP ~20° and start again.

In [ ]:
null_run = live("hold", seconds=600, dark_v=dark_v, note="null hunt: iterate analyzer HWP and QWP")  # ■ when done

<a id="slide"></a>
### Slide Test

**Question:** Does turning the analyzer slide the fringe without changing its depth, and can we place the peak and trough inside the window?

**Prediction:** The fringe moves by $4\varphi V_\pi/\pi$ for an analyzer-HWP turn $\varphi$; peak and trough levels stay constant. Target: peak near −0.40 V and trough near +0.40 V tee ("HIT").

**How:** Turn the analyzer HWP in 5° steps, waiting two redraws each. Watch the trough level as it moves.

**What could fool us:**
- A minimum at the window edge is the drive turning around, not the trough; only an interior minimum counts.
- $\Gamma_0$ drifts: re-check just before the $V_\pi$ run.

**Notable Decisions:**
- (A3) HIT = both bin extrema away from the two end bins (A1). Steering only; $V_\pi$ comes from the EOM notebook's run.

In [ ]:
slide_run = live("sine", seconds=600, dark_v=dark_v, note="slide test: analyzer HWP in 5 deg steps")  # ■ at HIT

In [ ]:
# Raw look: every sample of the last saved buffer, plus the peak/trough levels logged along the slide.
meta = json.loads((slide_run / "metadata.json").read_text())
with np.load(slide_run / "raw.npz") as z:
    pd, tee = z[meta["captures"][-1]["name"]] / 8192 * np.array([[rp_full_scale_v["in1"]], [rp_full_scale_v["in2"]]])
fig, (xy, levels) = plt.subplots(1, 2, figsize=(12, 4.5))
xy.plot(tee, pd, ",", color="k")
xy.set(xlabel="OUT1 at tee, IN2 raw (V)", ylabel="PD, IN1 raw (V)", title=f"last buffer ({meta['captures'][-1]['name']}), raw")
for key, colour, lab in (("peak", "tab:red", "peak"), ("trough", "tab:blue", "trough")):
    rows = [r for r in meta["log"] if f"{key} inside" in r["verdict"] or r["verdict"].startswith("HIT")]
    levels.plot([r["t_s"] for r in rows], [r[f"{key}_pd_v"] * 1e3 for r in rows], ".", color=colour, label=f"{lab} (only when inside)")
levels.set(xlabel="Time (s)", ylabel="Bin-mean level, A1 (mV)", title="Slide without shrink? (full scale from 0)", ylim=(-10, None))
levels.legend(fontsize=8)
fig.suptitle(f"{slide_run.name}"); plt.tight_layout(); plt.show()

**Decisions**

| # | Decision | Status |
|---|---|---|
| A1 | Peak/trough markers from 20 mV tee bins (≥ 5 samples), display aid only | proposed |
| A2 | Dark = mean of all raw buffers of a fully blocked run | proposed |
| A3 | HIT = both bin extrema away from the two end bins | proposed |

<a id="record"></a>
## Record (2026-11-04)

First alignment, done with the stand-alone prototype of this notebook (`experiments/high_speed_polarimeter/2026-11-04/sources/qwp_align.py`, same logic); runs in `experiments/high_speed_polarimeter/2026-11-04/` (`runs/` and `superseded_runs_2026-11-04.tar.xz`):

| Step | Run | Observation (raw unless noted) |
|---|---|---|
| Dark | `runs/190929_dark-level` (qwp_align_hold_20261104T190929Z) | −6.0 to −6.4 mV, beam blocked |
| Null hunt | qwp_align_hold_20261104T191054Z (in the superseded archive) | minimum +0.2 to +0.4 mV (≈ 6.5 mV above dark, ~3 % of the swing) |
| Slide test | `runs/191624_senarmont-slide` (qwp_align_sine_20261104T191624Z) | peak 176–180 mV and trough 0.1–2.2 mV whenever inside (A1 bins) across more than one fringe period; ended at HIT, peak ≈ −0.39 V, trough ≈ +0.41 V tee |

Final dials (operator): QWP 130°, analyzer HWP 230°, input HWP 300°. Next: $V_\pi$ run in the EOM notebook at this setting.